In [ ]:
from datetime import date
from enum import Enum
from typing import Self
from uuid import UUID, uuid4

In [ ]:
from pydantic import BaseModel, EmailStr, ConfigDict, Field, field_validator, model_validator

In [ ]:
class Depart(Enum):
    HR="HR"
    SALES="SALES"
    IT = "IT"
    ENGINEER="ENGINEER"

class Employee(BaseModel):
    model_config = ConfigDict(populate_by_name=True)
    
    e_id: UUID = Field(default_factory=uuid4, frozen=True)
    name: str = Field(min_length=1, max_length=12, frozen=True)
    email: EmailStr = Field(pattern=r".+@ilm\.com$", frozen=True)
    DOB: date = Field(alias="birth_date", repr=False, frozen=True)
    salary: float = Field(alias="Compensation", gt=0, frozen=True)
    department: Depart
    elected_benefit: bool

    @field_validator("DOB")
    @classmethod
    def check_age(cls, DOB: date) -> date:
        today = date.today()
        eighteenyrago= date(today.year-18, today.month, today.day)
        if DOB > eighteenyrago:
            raise ValueError("Employee must by above 18")
        return DOB


    @model_validator(mode="after")
    def check_it_benefit(self) -> Self:
        department = self.department
        electedbenefit = self.elected_benefit
        if department == Depart.IT and electedbenefit:
            raise ValueError("IT employee are not qualify for elected benefit")
        return self

In [ ]:
try:
    emps=Employee(name="Shafin",email="shafi@ilm.com", DOB="1984-10-10", salary=23_22.22, department="IT",elected_benefit=True)
except Exception as e:
    print(e)

In [ ]:
emps